In [ ]:
import os
from openai import OpenAI, AzureOpenAI
from google.colab import userdata

# OpenAI setup
OPENAI_API_KEY = userdata.get('Tradutor')
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# DeepSeek setup
DEEPSEEK_API_KEY = userdata.get('Tradutor2')
deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com/v1')

# Azure OpenAI setup
AZURE_KEY = userdata.get('AZURE_OPENAI_KEY')
AZURE_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
AZURE_VERSION = "2024-02-01"
azure_client = AzureOpenAI(
    api_key=AZURE_KEY,
    api_version=AZURE_VERSION,
    azure_endpoint=AZURE_ENDPOINT
)

print("OpenAI, DeepSeek, and Azure OpenAI clients initialized.")

In [ ]:
def translate_text_with_api(text, target_language, api_client):
    try:
        if api_client == openai_client:
            model_to_use = "gpt-3.5-turbo"
        elif api_client == deepseek_client:
            model_to_use = "deepseek-chat"
        elif isinstance(api_client, AzureOpenAI):
            # Azure uses deployment name instead of model name
            model_to_use = userdata.get('AZURE_OPENAI_DEPLOYMENT')
        else:
            return "An error occurred: Invalid API client provided."

        response = api_client.chat.completions.create(
            model=model_to_use,
            messages=[
                {"role": "system", "content": f"You are a highly accurate translation assistant. Translate the following text into {target_language}."},
                {"role": "user", "content": text}
            ],
            temperature=0.7,
            max_tokens=200
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred during translation: {e}"

print("Translation function updated to support Azure, OpenAI, and DeepSeek.")

In [ ]:
article_content = "Olá, como você está? Espero que esteja tendo um ótimo dia."
target_language = "English"

# Testando com Azure agora
if article_content and target_language:
    print("--- Tentando Tradução com Azure ---")
    azure_translated = translate_text_with_api(article_content, target_language, azure_client)
    print(f"Original text: {article_content}")
    print(f"Translated (Azure): {azure_translated}")
else:
    print("Please ensure 'article_content' and 'target_language' variables are set.")

In [ ]:
article_content = "Olá, como você está? Espero que esteja tendo um ótimo dia."
target_language = "English"

if article_content and target_language:
    deepseek_translated_text = translate_text_with_api(article_content, target_language, deepseek_client)
    print(f"Original text:\n{article_content}\n")
    print(f"Translated to {target_language} (DeepSeek):\n{deepseek_translated_text}")
else:
    print("Please ensure 'article_content' and 'target_language' variables are set.")

In [ ]:
def check_quotas():
    print('--- Checking OpenAI Status ---')
    try:
        openai_client.models.list()
        print('OpenAI: API Key is valid and reachable.')
        # Attempt a tiny completion
        openai_client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role': 'user', 'content': 'hi'}],
            max_tokens=1
        )
        print('OpenAI: Success (Quota available).')
    except Exception as e:
        print(f'OpenAI Error: {e}')

    print('\n--- Checking DeepSeek Status ---')
    try:
        # DeepSeek usually uses the same OpenAI-compatible list models call
        deepseek_client.models.list()
        print('DeepSeek: API Key is valid and reachable.')
        deepseek_client.chat.completions.create(
            model='deepseek-chat',
            messages=[{'role': 'user', 'content': 'hi'}],
            max_tokens=1
        )
        print('DeepSeek: Success (Balance available).')
    except Exception as e:
        print(f'DeepSeek Error: {e}')

check_quotas()